# Automação RPA - Extração de Clientes e Envio para o n8n

In [ ]:
import requests
from bs4 import BeautifulSoup

URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"

# 1) Baixar a página
response = requests.get(URL_PAGINA)
html = response.text

# 2) Ler a tabela de clientes
soup = BeautifulSoup(html, "html.parser")
linhas = soup.select("#clientes tbody tr")

clientes = []

for linha in linhas:
    colunas = linha.find_all("td")

    # Limpeza básica do valor de saldo para converter em número decimal (float)
    saldo_texto = colunas[2].get_text(strip=True)
    saldo_limpo = (
        saldo_texto.replace("R$", "")
        .replace(" ", "")
        .replace(".", "")
        .replace(",", ".")
    )

    cliente = {
        "nome": colunas[0].get_text(strip=True),
        "email": colunas[1].get_text(strip=True),
        "saldo": float(saldo_limpo) if saldo_limpo else 0.0,
        "perfil": colunas[3].get_text(strip=True),
    }

    clientes.append(cliente)

# 3) Mostrar o resultado no Colab
print(f"Clientes encontrados ({len(clientes)}):")
for c in clientes:
    print(c)

# 4) Enviar a lista para o Webhook do n8n
N8N_WEBHOOK = "https://eliannecarvalho.app.n8n.cloud/webhook-test/clientes"

# Faz o envio via POST
resposta_n8n = requests.post(N8N_WEBHOOK, json={"clientes": clientes})

print("\n--- Status do Envio para o n8n ---")
print("Status Code:", resposta_n8n.status_code)
print("Resposta do n8n:", resposta_n8n.text)
